-- 테스트 문제

In [1]:
print(3+5) # 더하기

8


In [3]:
a = 1
for i in range(a,11):
    print(i,end=' ')
    

1 2 3 4 5 6 7 8 9 10 

In [2]:
import pandas as pd

# 1. 데이터 읽기 (인코딩은 환경에 따라 'utf-8' 또는 'cp949' 적용)
df = pd.read_csv('greenData/pm10.csv', encoding='cp949')

# 2. '구분'이 '평균'인 행 제외 (실제 구 데이터만 추출)
# 팁: 학생들에게 조건부 인덱싱을 활용해 전처리하는 방법을 상기시켜주기 좋습니다.
df_districts = df[df['구분'] != '평균']

# 3. 다중 조건 필터링: PM10 >= 25 AND PM25 >= 15
# 팁: 비트 연산자(&) 사용과 괄호()의 중요성을 강조할 수 있습니다.
condition = (df_districts['미세먼지(PM10)'] >= 25) & (df_districts['초미세먼지(PM25)'] >= 15)
df_warning = df_districts[condition]

# 4. 특정 컬럼만 선택하여 출력
result = df_warning[['구분', '미세먼지(PM10)', '초미세먼지(PM25)']]
print("--- 미세먼지 및 초미세먼지 주의 지역 ---")
print(result)

--- 미세먼지 및 초미세먼지 주의 지역 ---
          구분  미세먼지(PM10)  초미세먼지(PM25)
5        관악구        25.0         17.0
9        노원구        28.0         16.0
14      서대문구        31.0         16.0
15       서초구        31.0         15.0
19       양천구        25.0         15.0
...      ...         ...          ...
228377  영등포구        32.0         26.0
228378   용산구        35.0         25.0
228380   종로구        28.0         26.0
228381    중구        26.0         26.0
228382   중랑구        28.0         23.0

[100203 rows x 3 columns]


In [7]:
import pandas as pd

# 1. 데이터 읽기 (공공데이터는 주로 cp949 또는 euc-kr 인코딩 사용)
# 파일 경로 및 인코딩은 실제 파일에 맞게 수정하여 안내하시면 됩니다.
df = pd.read_csv('greenData/accident.csv', encoding='cp949')

# 2. datetime 변환 및 '사고월' 파생 변수 생성
df['사고일자'] = pd.to_datetime(df['사고일자'])
df['사고월'] = df['사고일자'].dt.month  # dt 접근자를 사용하여 월(month)만 추출

# 3. 월별 사고 건수 집계 (인덱스 정렬 포함)
monthly_count = df['사고월'].value_counts().sort_index()
print("--- 월별 사고 발생 건수 ---")
print(monthly_count)
print(f"가장 사고가 많은 월: {monthly_count.idxmax()}월\n")

# 4. '사상자수' 파생 변수 생성 (사망 + 부상)
df['사상자수'] = df['사망'] + df['부상']

# 5. 노선명별 사상자수 합계 및 상위 5개 추출
# 팁: 학생들에게 groupby 이후 sum() 연산, 그리고 sort_values() 연결(Chaining) 기법을 강조하기 좋습니다.
route_casualties = df.groupby('노선명')['사상자수'].sum()
top5_routes = route_casualties.sort_values(ascending=False).head(5)

print("--- 사상자수 기준 상위 5개 위험 노선 ---")
print(top5_routes, "\n")

# 6. 다중 조건 필터링 (원인 == '졸음' AND 사망 >= 1)
drowsy_fatal_accidents = df[(df['원인'] == '졸음') & (df['사망'] >= 1)]
fatal_count = len(drowsy_fatal_accidents)

print(f"--- '졸음' 운전 중 사망자가 발생한 치명적 사고 건수: {fatal_count}건 ---")

--- 월별 사고 발생 건수 ---
사고월
1     349
2     326
3     384
4     418
5     400
6     405
7     413
8     429
9     463
10    380
11    378
12    366
Name: count, dtype: int64
가장 사고가 많은 월: 9월

--- 사상자수 기준 상위 5개 위험 노선 ---
노선명
경부선      620
중부선      293
중부내륙선    279
영동선      274
서해안선     274
Name: 사상자수, dtype: int64 

--- '졸음' 운전 중 사망자가 발생한 치명적 사고 건수: 88건 ---


In [11]:
import pandas as pd

# 데이터 로드
hospital = pd.read_csv('greenData/hospital.csv', encoding='cp949')
pm10 = pd.read_csv('greenData/pm10.csv', encoding='cp949')

# [문제 1] 병합을 위한 공통 키(Key) 생성
# '인천병원' -> '인천'으로 변경하여 pm10 데이터의 '구분' 컬럼과 형식을 맞춤
hospital['지역'] = hospital['소속기관'].str.replace('병원', '')
pm10.rename(columns={'구분': '지역'}, inplace=True) # 컬럼명을 동일하게 변경

print("--- 병원 데이터 지역 컬럼 생성 결과 ---")
print(hospital[['소속기관', '지역']].head())

--- 병원 데이터 지역 컬럼 생성 결과 ---
   소속기관  지역
0  인천병원  인천
1  인천병원  인천
2  인천병원  인천
3  인천병원  인천
4  인천병원  인천


In [12]:
# [문제 2] 데이터 병합 (Left Join)
# 병원 데이터를 기준으로 미세먼지 정보를 붙임
merged_df = pd.merge(hospital, pm10, on='지역', how='left')

print("\n--- 데이터 병합 결과 (상단 5행) ---")
print(merged_df.head())


--- 데이터 병합 결과 (상단 5행) ---
     연도  소속기관       협력병원명     업무협력 기간         업무협력 내용  지역   일시  미세먼지(PM10)  \
0  2024  인천병원       인하대병원  2006-10-현재  진료의뢰 및 회송 관련 등  인천  NaN         NaN   
1  2024  인천병원      가천대길병원  2005-04-현재  진료의뢰 및 회송 관련 등  인천  NaN         NaN   
2  2024  인천병원  고려대학교 구로병원  2008-03-현재  진료의뢰 및 회송 관련 등  인천  NaN         NaN   
3  2024  인천병원     건국대학교병원  2013-10-현재  진료의뢰 및 회송 관련 등  인천  NaN         NaN   
4  2024  인천병원     강동경희대병원  2015-03-현재  진료의뢰 및 회송 관련 등  인천  NaN         NaN   

   초미세먼지(PM25)  
0          NaN  
1          NaN  
2          NaN  
3          NaN  
4          NaN  


In [13]:
# [문제 3] 결측치 확인 및 처리
# 1. 결측치 개수 확인
print("\n--- 결측치 개수 확인 ---")
print(merged_df.isnull().sum())

# 2. 미세먼지(PM10) 농도의 결측치를 평균값으로 채우기
avg_pm10 = merged_df['미세먼지(PM10)'].mean()
merged_df['미세먼지(PM10)'] = merged_df['미세먼지(PM10)'].fillna(avg_pm10)

# 3. 결측치 처리 후 결과 확인
print(f"\n--- 결측치 채우기 완료 (평균값: {avg_pm10:.2f}) ---")
print(merged_df.isnull().sum())


--- 결측치 개수 확인 ---
연도               0
소속기관             0
협력병원명            0
업무협력 기간          0
업무협력 내용          0
지역               0
일시             608
미세먼지(PM10)     608
초미세먼지(PM25)    608
dtype: int64

--- 결측치 채우기 완료 (평균값: nan) ---
연도               0
소속기관             0
협력병원명            0
업무협력 기간          0
업무협력 내용          0
지역               0
일시             608
미세먼지(PM10)     608
초미세먼지(PM25)    608
dtype: int64
